In [0]:
from TornAPI.Torn import Faction
from pyspark.sql.functions import lit, explode, from_unixtime, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, ArrayType

import datetime as dt

faction_api = Faction(dbutils.secrets.get("Personal", "TornAPI"))

In [0]:
if spark.catalog.tableExists("torn.faction.armouryUsage"):
    current_data = spark.read.table("torn.faction.armouryUsage")
    max_date = current_data.select("timestamp").agg({"timestamp":"max"}).collect()
    current_max_date = max_date[0]["max(timestamp)"]
    id_list = [id_num[0] for id_num in current_data.select("id").collect()]
else:
    current_max_date = 1681484237
    id_list = []

In [0]:
while current_max_date <= (dt.datetime.today() + dt.timedelta(days=-1)).timestamp():
    data = faction_api.get_news_armoury_action(ts_from=current_max_date, sort="ASC")
    sp_armoury = spark.createDataFrame(data["news"])

    sp_armoury.write.format("delta").mode("append").saveAsTable("torn.faction.armouryUsage")
    
    current_data = spark.read.table("torn.faction.armouryUsage")
    max_date = current_data.select("timestamp").agg({"timestamp":"max"}).collect()
    current_max_date = max_date[0]["max(timestamp)"]
